In [1]:
from collections import Counter
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from imblearn.pipeline import make_pipeline
from sklearn.metrics import classification_report

# Random forest classifier

In [2]:
test_data = pd.read_csv("/ibex/user/sotoorda/masterh1/LLM project/Random_forest/Objects/test_cell_data.csv")
ref_data = pd.read_csv("/ibex/user/sotoorda/masterh1/LLM project/Random_forest/Objects/ref_cell_data.csv")
"/ibex/user/sotoorda/masterh1/LLM project/Random_forest/test_cell_data.csv"
counter_data = Counter(ref_data["CellType"])
counter_test = Counter(test_data["CellType"])

print(counter_data)
print(counter_test)

Counter({'Naive T-cell': 25266, 'Neutrophil': 12216, 'Erythroblast': 8226, 'Immature-Neutrophil': 7981, 'Follicular B cell': 6041, 'CD8 T-cell': 5283, 'NK cells': 5123, 'Dendritic Cell': 4131, 'Eosinophil': 3804, 'Monocyte': 3531, 'Early-Erythroblast': 2685, 'HSC': 2276, 'MPP': 1988, 'Plasma Cell': 1906, 'PreB': 1882, 'ERP': 1860, 'ProB': 1471, 'GMP': 1151, 'Granulocytic-UNK': 1093, 'Pre-Dendritic': 973, 'pre-PC': 821, 'MKP': 404, 'MDP': 379, 'MEP': 341, 'CLP': 308, 'Stromal': 181, 'Eo/B/Mast': 119, 'pre-T': 82, 'Platelet': 65})
Counter({'ProB': 13226, 'HSC': 2965, 'PreB': 2790, 'MPP': 1488, 'GMP': 1335, 'CLP': 554, 'MDP': 330, 'ERP': 288, 'MEP': 261, 'Eo/B/Mast': 229})


## Preparing data for training

In [3]:
# Prepare the data for training and testing
X_train = ref_data.drop(columns=["CellType", "CellName"])  # Features (gene expression)
y_train = ref_data["CellType"] # Target (cell type)

X_test = test_data.drop(columns=["CellType", "CellName"]) 
y_test = test_data["CellType"]

# Ensure train/test features are identical and in the same order
common_features = [c for c in X_train.columns if c in X_test.columns]
X_train = X_train[common_features]
X_test = X_test[common_features]

print(f"Aligned feature count: {len(common_features)}")

Aligned feature count: 602


## Running the classifier

In [4]:
pipeline = make_pipeline(
    RandomForestClassifier(n_estimators=150, random_state=42)
)
pipeline.fit(X_train, y_train)

# Classify and report the results
print(classification_report(y_test, pipeline.predict(X_test)))

# Full probabilistic output for each class
y_prob = pipeline.predict_proba(X_test)

/ibex/user/sotoorda/conda-environments/test_env/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/ibex/user/sotoorda/conda-environments/test_env/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/ibex/user/sotoorda/conda-environments/test_env/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metr

                     precision    recall  f1-score   support

         CD8 T-cell       0.00      0.00      0.00         0
                CLP       0.54      0.01      0.02       554
     Dendritic Cell       0.00      0.00      0.00         0
                ERP       0.72      0.76      0.74       288
 Early-Erythroblast       0.00      0.00      0.00         0
          Eo/B/Mast       1.00      0.03      0.05       229
       Erythroblast       0.00      0.00      0.00         0
  Follicular B cell       0.00      0.00      0.00         0
                GMP       0.95      0.25      0.40      1335
   Granulocytic-UNK       0.00      0.00      0.00         0
                HSC       0.61      0.86      0.72      2965
Immature-Neutrophil       0.00      0.00      0.00         0
                MDP       1.00      0.07      0.14       330
                MEP       0.61      0.07      0.13       261
                MKP       0.00      0.00      0.00         0
                MPP    

## Getting prediction DF

In [5]:
classes = pipeline.named_steps['randomforestclassifier'].classes_
prob_df = pd.DataFrame(y_prob, columns=classes)
y_pred = classes[np.argmax(y_prob, axis=1)]
max_prob = y_prob.max(axis=1)

results_df = pd.DataFrame({
    "Cell_id": test_data["CellName"].values,
    "Ground_truth": y_test.values,
    "Predicted_label": y_pred,
    "Prediction_prob": max_prob
})

results_df = pd.concat([results_df, prob_df], axis=1)
results_df.to_csv("/ibex/user/sotoorda/masterh1/LLM project/Random_forest/prediction_results.csv", index=False)